## Install LiteLLM

In [1]:
!pip install litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: py

## Define API endpoint and load tokenizer.

In [2]:
from transformers import AutoTokenizer

api = "[REDACTED]" # vLLM API endpoint here
model_name = "Qwen/Qwen3.6-27B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## Load the dataset for compression.

In [3]:
import re
import random
from datasets import load_dataset

# Load only 7 files (~100k samples) from Dolci Think
random.seed(0)
files = random.sample(range(156), 7)
files = [f"https://huggingface.co/datasets/allenai/Dolci-Think-SFT-7B/resolve/main/data/train-{i:05d}-of-00156.parquet" for i in files]
dataset = load_dataset("parquet", data_files=files, split="train")

# Keep only the first turn of conversations
dataset = dataset.map(lambda x: {"messages": x["messages"][:2]})

# Keep only samples where assistant has <think>...</think> with non-whitespace inside
dataset = dataset.filter(
  lambda x: (
    len(x["messages"]) >= 2
    and x["messages"][0]["role"] == "user"
    and x["messages"][1]["role"] == "assistant"
    and (m := re.search(r"<think>(.*?)</think>", x["messages"][1]["content"], flags=re.DOTALL)) is not None
    and m.group(1).strip() != ""
  )
)

# Keep only samples with total token count <= 1000
def keep(batch):
  texts = [
    m[0]["content"] + m[1]["content"]
    for m in batch["messages"]
  ]
  lengths = tokenizer(
    texts,
    add_special_tokens = False,
    return_length = True,
  )["length"]
  return [length <= 1000 for length in lengths]

dataset = dataset.filter(
  keep,
  batched = True,
  batch_size = 1000,
  num_proc = 4,
)

# Keep only 3000 samples
dataset = dataset.shuffle(seed=0).select(range(3000))

print(dataset)

data/train-00098-of-00156.parquet:   0%|          | 0.00/33.2M [00:00<?, ?B/s]

data/train-00107-of-00156.parquet:   0%|          | 0.00/77.2M [00:00<?, ?B/s]

data/train-00010-of-00156.parquet:   0%|          | 0.00/295M [00:00<?, ?B/s]

data/train-00066-of-00156.parquet:   0%|          | 0.00/379M [00:00<?, ?B/s]

data/train-00130-of-00156.parquet:   0%|          | 0.00/113M [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/101775 [00:00<?, ? examples/s]

Filter:   0%|          | 0/101775 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/101755 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'dataset_source', 'id'],
    num_rows: 3000
})


### Apply few-shot prompt.

In [4]:
from datasets import load_dataset

template = """\
Simplify the following THOUGHT PROCESS, only including the key information in each step.
Do NOT add additional information that is not included in the original THOUGHT PROCESS.
Match the style and flow of the original THOUGHT PROCESS.
Respond with ONLY the SIMPLIFIED THOUGHT PROCESS and no other text.

Example THOUGHT PROCESS:
Okay, I need to write a Python program that calculates the area of a circle given its radius. Let me think about the steps here.

First, the program has to read input from standard input. The input is a floating-point number, which is the radius. So I'll use input() function to get that. But since it's a float, I have to convert it. Like, radius = float(input()), right?

Then, the formula for the area is π times radius squared. Oh right, I need to import the math module to get π. So I should import math and use math.pi. Alternatively, maybe from math import pi? But to be safe, maybe just use math.pi.

So the area would be math.pi * (radius ** 2). Then I need to output that. The output should be a floating-point number. So print the area with print.

Wait, but how does the output look exactly? The problem says to output a single floating-point number. So just print(area) should work, since the calculation would already be a float.

Let me check possible issues. What if the input is something like 0? Then the area is zero, which is okay. And negative radius? Hmm, the problem probably assumes a valid input, so maybe we don't have to handle that. The problem statement says the input is the radius, so maybe it's okay to proceed.

Putting it all together:

First line: import math.

Then read the radius: r = float(input())

Calculate area: area = math.pi * r**2

Then print(area).

Wait, but when I print, maybe I should format it to ensure it's a float? Like, if the user enters an integer as input, but the output must be a float. Well, the calculation will result in a float anyway, so printing it directly should be okay. Let me see. For example, if radius is 2, area would be math.pi * 4, which is like 12.566..., so printed as a float.

So the code structure would be:

import math

r = float(input())
print(math.pi * r ** 2)

Wait, that's even more concise. Because we can do it in two lines. No need to store the area in a variable, unless necessary. The problem doesn't require that, so this should work.

Hmm, but sometimes when you have to output a float, maybe there's a case where the result is an integer? Like radius 0, then it's 0.0? But in Python, 0.0 is still a float. So the output would be correct as is.

Testing with the input 5, the area is π*25 ≈ 78.5398163397..., so the program would print that. So this should be correct.

I think that's all. Let me write the code as such.

Example SIMPLIFIED THOUGHT PROCESS:
I need to write a Python program that calculates the area of a circle given its radius.

First, I need to read the radius from standard input. Since it's a float, I need to convert it. So radius = float(input()).

The formula for the area is π times radius squared. I need to import math first then use math.pi. So area = math.pi * (radius ** 2).

Then, since the output should be a float and calculation results in a float, I just need to print that value with print(area).

Let me check for possible issues. Input is 0.0? then area is 0.0, which is okay. Input is negative? Problem probably assumes valid input. Input is integer? math.pi is a float so output will still be a float.

Example THOUGHT PROCESS:
The problem states that an equilateral triangle has a side length of 20. A square has the same perimeter as this triangle, and I need to find the area of the square. The options are 25, 400, 225, 60, and 100.

First, since it's an equilateral triangle, all sides are equal. So, each side is 20. The perimeter of the triangle is three times the side length. So, perimeter P_triangle = 3 * 20 = 60.

Now, the square has the same perimeter, so perimeter of square P_square = 60.

A square has four equal sides, so each side of the square is the perimeter divided by 4. So, side length s = 60 / 4 = 15.

The area of a square is side squared, so area A = s^2 = 15^2 = 225.

Looking at the options, 225 is one of them, which is choice (C).

I should double-check. Triangle perimeter: 3 sides * 20 = 60. Square perimeter: 60. Each side of square: 60 / 4 = 15. Area: 15 * 15 = 225. Yeah, that seems correct.

Option (B) is 400, which is 20 squared, but that's not relevant here. Option (A) 25 is 5 squared, (D) 60 is the perimeter, not area, (E) 100 is 10 squared. So, 225 makes sense.

The triangle is equilateral with side 20, perimeter 60. Square same perimeter, side 15, area 225. I think that's it.

Example SIMPLIFIED THOUGHT PROCESS:
I need to find the area of a square whose perimeter is equal to an equilateral triangle.

The triangle has a side length of 20, and since it's equilateral, the perimeter is 3 * 20 = 60.

So the square has the same perimeter of 60.

A square has four equal sides. So, the square's side length is 60 / 4 = 15.

The area of a square is side squared, so 15^2 = 225.

In the given options, 225 is choice (C).

Your THOUGHT PROCESS:
{}

Your SIMPLIFIED THOUGHT PROCESS:
"""

def extract_cot(messages):
    assistant = next(m["content"] for m in messages if m["role"] == "assistant")
    match = re.search(r"<think>(.*?)</think>", assistant, flags=re.DOTALL)
    return match.group(1).strip() if match and match.group(1).strip() else ""

def formatting_func(batch):
  texts = []
  lengths = []
  for messages in batch["messages"]:
    cot = extract_cot(messages)
    text = template.format(cot)
    texts.append(text)
    lengths.append(len(tokenizer.tokenize(text)))
  return {"text": texts, "length": lengths}

dataset = dataset.map(
  formatting_func,
  batched = True
)

print(dataset)
print("Max sequence length in dataset:", max(dataset["length"]))

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'dataset_source', 'id', 'text', 'length'],
    num_rows: 3000
})
Max sequence length in dataset: 2344


In [5]:
print(dataset["text"][0])

Simplify the following THOUGHT PROCESS, only including the key information in each step.
Do NOT add additional information that is not included in the original THOUGHT PROCESS.
Match the style and flow of the original THOUGHT PROCESS.
Respond with ONLY the SIMPLIFIED THOUGHT PROCESS and no other text.

Example THOUGHT PROCESS:
Okay, I need to write a Python program that calculates the area of a circle given its radius. Let me think about the steps here.

First, the program has to read input from standard input. The input is a floating-point number, which is the radius. So I'll use input() function to get that. But since it's a float, I have to convert it. Like, radius = float(input()), right?

Then, the formula for the area is π times radius squared. Oh right, I need to import the math module to get π. So I should import math and use math.pi. Alternatively, maybe from math import pi? But to be safe, maybe just use math.pi.

So the area would be math.pi * (radius ** 2). Then I need to o

## Test compression on 1 sample.

In [6]:
from litellm import completion

prompt = dataset["text"][0]
output = completion(
    model=f"huggingface/{model_name}",
    api_base = api,
    temperature = 0.0,
    max_tokens = 3500,
    messages = [{"role": "user", "content": prompt}],
    extra_body = {"chat_template_kwargs": {"enable_thinking": False}}
).choices[0].message.content

print(output)

I need to find the area of a square whose perimeter is equal to an equilateral triangle.

The triangle has a side length of 20, and since it's equilateral, the perimeter is 3 * 20 = 60.

So the square has the same perimeter of 60.

A square has four equal sides. So, the square's side length is 60 / 4 = 15.

The area of a square is side squared, so 15^2 = 225.

In the given options, 225 is choice (C).


## Batched inference through the dataset.

In [7]:
from litellm import batch_completion

batch_size = 100
short_cots = []
length_long = 0
length_short = 0

for i in range(0, len(dataset), batch_size):
  batchset = dataset.select(range(i, min(i + batch_size, len(dataset))))
  messages = [[{"role": "user", "content": sample["text"]}] for sample in batchset]
  outputs = batch_completion(
    model=f"huggingface/{model_name}",
    api_base = api,
    temperature = 0.0,
    max_tokens = 3500,
    messages = messages,
    extra_body = {"chat_template_kwargs": {"enable_thinking": False}}
  )
  for j, out in enumerate(outputs):
    short_cot = out.choices[0].message.content
    short_cots.append(short_cot)
    length_short += len(tokenizer.tokenize(short_cot))
    long_cot = extract_cot(batchset["messages"][j])
    length_long += len(tokenizer.tokenize(long_cot))

  print(f"Batch {(i // batch_size) + 1} ({i + len(batchset)} / {len(dataset)}):")
  print(f">> Mean Length Long CoT  : {length_long  / (i + len(batchset))}")
  print(f">> Mean Length Short CoT : {length_short / (i + len(batchset))}\n")

Batch 1 (100 / 3000):
>> Mean Length Long CoT  : 338.65
>> Mean Length Short CoT : 141.7

Batch 2 (200 / 3000):
>> Mean Length Long CoT  : 339.15
>> Mean Length Short CoT : 143.53

Batch 3 (300 / 3000):
>> Mean Length Long CoT  : 337.4033333333333
>> Mean Length Short CoT : 140.99333333333334

Batch 4 (400 / 3000):
>> Mean Length Long CoT  : 333.0875
>> Mean Length Short CoT : 140.77

Batch 5 (500 / 3000):
>> Mean Length Long CoT  : 333.24
>> Mean Length Short CoT : 140.818

Batch 6 (600 / 3000):
>> Mean Length Long CoT  : 331.1616666666667
>> Mean Length Short CoT : 141.025

Batch 7 (700 / 3000):
>> Mean Length Long CoT  : 331.2457142857143
>> Mean Length Short CoT : 140.89285714285714

Batch 8 (800 / 3000):
>> Mean Length Long CoT  : 330.98125
>> Mean Length Short CoT : 141.4575


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/

AttributeError: 'APIError' object has no attribute 'choices'

### Continue after error.

In [8]:
i = 800
short_cots = short_cots[:i]
print(len(short_cots))

batch_size = 100

for i in range(i, len(dataset), batch_size):
  batchset = dataset.select(range(i, min(i + batch_size, len(dataset))))
  messages = [[{"role": "user", "content": sample["text"]}] for sample in batchset]
  outputs = batch_completion(
    model=f"huggingface/{model_name}",
    api_base = api,
    temperature = 0.0,
    max_tokens = 3500,
    messages = messages,
    extra_body = {"chat_template_kwargs": {"enable_thinking": False}}
  )
  for j, out in enumerate(outputs):
    short_cot = out.choices[0].message.content
    short_cots.append(short_cot)
    length_short += len(tokenizer.tokenize(short_cot))
    long_cot = extract_cot(batchset["messages"][j])
    length_long += len(tokenizer.tokenize(long_cot))

  print(f"Batch {(i // batch_size) + 1} ({i + len(batchset)} / {len(dataset)}):")
  print(f">> Mean Length Long CoT  : {length_long  / (i + len(batchset))}")
  print(f">> Mean Length Short CoT : {length_short / (i + len(batchset))}\n")

800
Batch 9 (900 / 3000):
>> Mean Length Long CoT  : 333.1377777777778
>> Mean Length Short CoT : 141.67777777777778

Batch 10 (1000 / 3000):
>> Mean Length Long CoT  : 332.142
>> Mean Length Short CoT : 140.941

Batch 11 (1100 / 3000):
>> Mean Length Long CoT  : 330.17545454545456
>> Mean Length Short CoT : 140.36727272727273

Batch 12 (1200 / 3000):
>> Mean Length Long CoT  : 328.8983333333333
>> Mean Length Short CoT : 140.7075

Batch 13 (1300 / 3000):
>> Mean Length Long CoT  : 329.35769230769233
>> Mean Length Short CoT : 141.39538461538461

Batch 14 (1400 / 3000):
>> Mean Length Long CoT  : 330.01785714285717
>> Mean Length Short CoT : 141.3007142857143

Batch 15 (1500 / 3000):
>> Mean Length Long CoT  : 328.706
>> Mean Length Short CoT : 141.11733333333333

Batch 16 (1600 / 3000):
>> Mean Length Long CoT  : 328.13125
>> Mean Length Short CoT : 141.1575

Batch 17 (1700 / 3000):
>> Mean Length Long CoT  : 327.61411764705883
>> Mean Length Short CoT : 141.01823529411766

Batch 18 (

KeyboardInterrupt: 

In [10]:
i = 2000
short_cots = short_cots[:i]
print(len(short_cots))

batch_size = 100

for i in range(i, len(dataset), batch_size):
  batchset = dataset.select(range(i, min(i + batch_size, len(dataset))))
  messages = [[{"role": "user", "content": sample["text"]}] for sample in batchset]
  outputs = batch_completion(
    model=f"huggingface/{model_name}",
    api_base = api,
    temperature = 0.0,
    max_tokens = 3500,
    messages = messages,
    extra_body = {"chat_template_kwargs": {"enable_thinking": False}}
  )
  for j, out in enumerate(outputs):
    short_cot = out.choices[0].message.content
    short_cots.append(short_cot)
    length_short += len(tokenizer.tokenize(short_cot))
    long_cot = extract_cot(batchset["messages"][j])
    length_long += len(tokenizer.tokenize(long_cot))

  print(f"Batch {(i // batch_size) + 1} ({i + len(batchset)} / {len(dataset)}):")
  print(f">> Mean Length Long CoT  : {length_long  / (i + len(batchset))}")
  print(f">> Mean Length Short CoT : {length_short / (i + len(batchset))}\n")

2000
Batch 21 (2100 / 3000):
>> Mean Length Long CoT  : 328.58142857142855
>> Mean Length Short CoT : 140.89380952380952

Batch 22 (2200 / 3000):
>> Mean Length Long CoT  : 329.17636363636365
>> Mean Length Short CoT : 141.05363636363637

Batch 23 (2300 / 3000):
>> Mean Length Long CoT  : 329.7182608695652
>> Mean Length Short CoT : 141.05739130434782

Batch 24 (2400 / 3000):
>> Mean Length Long CoT  : 329.97583333333336
>> Mean Length Short CoT : 141.16375

Batch 25 (2500 / 3000):
>> Mean Length Long CoT  : 329.8728
>> Mean Length Short CoT : 141.0484

Batch 26 (2600 / 3000):
>> Mean Length Long CoT  : 329.9680769230769
>> Mean Length Short CoT : 141.22884615384615

Batch 27 (2700 / 3000):
>> Mean Length Long CoT  : 329.2985185185185
>> Mean Length Short CoT : 140.96814814814815

Batch 28 (2800 / 3000):
>> Mean Length Long CoT  : 329.33785714285716
>> Mean Length Short CoT : 140.90178571428572

Batch 29 (2900 / 3000):
>> Mean Length Long CoT  : 329.7893103448276
>> Mean Length Short C

## Make class-conditioned dataset.

In [11]:
from datasets import Dataset

dataset = Dataset.from_dict(
  {
    'instruction': [messages[0]["content"] for messages in dataset["messages"]],
    'reasoning_long': [extract_cot(messages) for messages in dataset["messages"]],
    'reasoning_short': short_cots,
    'answer': [messages[1]["content"].split("</think>")[-1].strip() for messages in dataset["messages"]]
  }
)

print(dataset)
print(dataset[0])
print(dataset[-1])

Dataset({
    features: ['instruction', 'reasoning_long', 'reasoning_short', 'answer'],
    num_rows: 3000
})
{'instruction': 'An equilateral triangle has a side length of 20 . If a square has the same perimeter as this triangle, the area of the square is\n(A) 25\n(B) 400\n(C) 225\n(D) 60\n(E) 100', 'reasoning_long': "The problem states that an equilateral triangle has a side length of 20. A square has the same perimeter as this triangle, and I need to find the area of the square. The options are 25, 400, 225, 60, and 100.\n\nFirst, since it's an equilateral triangle, all sides are equal. So, each side is 20. The perimeter of the triangle is three times the side length. So, perimeter P_triangle = 3 * 20 = 60.\n\nNow, the square has the same perimeter, so perimeter of square P_square = 60.\n\nA square has four equal sides, so each side of the square is the perimeter divided by 4. So, side length s = 60 / 4 = 15.\n\nThe area of a square is side squared, so area A = s^2 = 15^2 = 225.\n\nL

## Save dataset to huggingface.

In [ ]:
dataset.push_to_hub("jeypiii/Dolci-Think-SFT-7B_C3oT_v3", split="train", token="[REDACTED]")